# 02 — Pipeline y Entrenamiento (Versión para completar en clase)

**Taller MLOps UNI** — Predicción de resistencia del concreto

En este notebook construiremos un Pipeline de Scikit-Learn que incluye
preprocesamiento y modelo, lo entrenaremos y lo exportaremos con `joblib`.

> **Instrucciones**: Completa las celdas marcadas con `# TODO` siguiendo
> las indicaciones del docente.

---

## 1. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Librerías cargadas.")

## 2. Cargar datos

In [ ]:
DATA_PATH = "../data/raw/concrete_data.csv"

df = pd.read_csv(DATA_PATH)
df.columns = df.columns.str.strip()
print(f"Dataset: {df.shape}")
df.head(3)

## 3. Separar features y target

El **target** es `concrete_compressive_strength`. Todas las demás columnas son **features**.

In [ ]:
# TODO: Define las features (X) y el target (y)
# Pista: X = df[___] , y = df[___]

FEATURES = [
    "cement", "blast_furnace_slag", "fly_ash", "water",
    "superplasticizer", "coarse_aggregate", "fine_aggregate", "age",
]
TARGET = "concrete_compressive_strength"

X = ___  # TODO: Selecciona las columnas de features
y = ___  # TODO: Selecciona la columna target

print(f"Features: {X.shape} | Target: {y.shape}")

## 4. Dividir en train y test

In [ ]:
# TODO: Usa train_test_split para dividir los datos
# Parámetros: test_size=0.2, random_state=42

X_train, X_test, y_train, y_test = ___  # TODO: Completa

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")

## 5. ¿Qué es el Train-Serve Skew?

**Problema**: Si escalamos los datos con `StandardScaler` de forma separada al modelo,
corremos el riesgo de olvidar aplicar la misma transformación en producción,
o de usar estadísticas (media, desviación estándar) diferentes.

**Solución**: Meter el `StandardScaler` y el `RandomForestRegressor` dentro de un
**Pipeline** de Scikit-Learn. Al exportar el Pipeline completo, las transformaciones
viajan junto con el modelo.

## 6. Crear el Pipeline

In [ ]:
# TODO: Crea un Pipeline con dos pasos:
#   1. ("scaler", StandardScaler())
#   2. ("model", RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42))

pipeline = Pipeline([
    ___,  # TODO: Paso 1 - Escalado
    ___,  # TODO: Paso 2 - Modelo
])

print("Pipeline creado:")
print(pipeline)

## 7. Entrenar el Pipeline

In [ ]:
# TODO: Entrena el pipeline con los datos de entrenamiento
# Pista: pipeline.fit(X_train, y_train)

___  # TODO: Entrena aquí

print("Modelo entrenado exitosamente.")

## 8. Evaluar el modelo

In [ ]:
# TODO: Genera predicciones sobre el conjunto de test
# Pista: y_pred = pipeline.predict(___)

y_pred = ___  # TODO: Predice aquí

# Calcular métricas
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE:  {mae:.4f}")
print(f"RMSE: {rmse:.4f}")
print(f"R²:   {r2:.4f}")

## 9. Exportar el modelo con joblib

Al exportar el **Pipeline completo**, estamos empaquetando:
- El `StandardScaler` (con la media y desviación estándar aprendidas)
- El `RandomForestRegressor` (con los árboles entrenados)

Esto **elimina el Train-Serve Skew** porque en producción solo necesitas
cargar un archivo y llamar `.predict()`.

In [ ]:
# TODO: Exporta el pipeline entrenado con joblib
# Pista: joblib.dump(pipeline, "ruta/al/archivo.joblib")

MODEL_PATH = "../models/modelo_concreto.joblib"

___  # TODO: Exporta aquí

print(f"Modelo exportado en: {MODEL_PATH}")

## 10. Verificar: cargar y predecir

Simulemos lo que haría un sistema en producción: cargar el modelo y predecir.

In [ ]:
# TODO: Carga el modelo exportado y haz una predicción de prueba
# Pista: modelo_cargado = joblib.load(MODEL_PATH)

modelo_cargado = ___  # TODO: Carga aquí

# Ejemplo: una mezcla de concreto
ejemplo = pd.DataFrame([{
    "cement": 540.0,
    "blast_furnace_slag": 0.0,
    "fly_ash": 0.0,
    "water": 162.0,
    "superplasticizer": 2.5,
    "coarse_aggregate": 1040.0,
    "fine_aggregate": 676.0,
    "age": 28,
}])

prediccion = modelo_cargado.predict(ejemplo)
print(f"Resistencia predicha: {prediccion[0]:.2f} MPa")

---

## ¡Felicidades!

Has completado el flujo de:
1. Cargar datos
2. Crear un Pipeline (evitando Train-Serve Skew)
3. Entrenar y evaluar
4. Exportar con joblib
5. Verificar la inferencia

**Siguiente paso**: Documenta tu modelo en la Model Card (`docs/model_card.md`)
y prepara el despliegue con Docker.